# Activation Patching Pipeline for useEffect Dependency Circuit Analysis

Run all 4 experiments over N counterfactual pairs and generate heatmaps.

### Usage
    1. Set MODEL_NAME and DATASET_PATH below.
    2. Run this script in a Jupyter notebook or as a Python file.
    3. Heatmaps are saved as PNG files in OUTPUT_DIR.

### Requirements
    pip install transformer-lens plotly kaleido


In [1]:
%pip install git+https://github.com/TransformerLensOrg/TransformerLens
%pip install huggingface_hub==1.11.0
%pip install kaleido

  Cloning https://github.com/TransformerLensOrg/TransformerLens to /tmp/pip-req-build-jv32ow4j
  Running command git clone --filter=blob:none --quiet https://github.com/TransformerLensOrg/TransformerLens /tmp/pip-req-build-jv32ow4j
  Resolved https://github.com/TransformerLensOrg/TransformerLens to commit 9e3f5d79bc76f702180c1c8c01a48d23a442bfed
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached torchvision-0.22.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached torch-2.7.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.6.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvi

In [2]:
import json
import torch
import numpy as np
import plotly.express as px
import plotly.io as pio
import transformer_lens.patching as patching
from transformer_lens.model_bridge import TransformerBridge
from pathlib import Path
from huggingface_hub import login

## Configuration

In [8]:
MODEL_NAME = "meta-llama/Llama-3.2-1B"  # Change to your model
DATASET_PATH = "pilot_dataset.json"
OUTPUT_DIR = Path("llama3_1b_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Only use substitution type (removal type may have token length mismatch)
USE_TYPES = ["substitution"]  # Add "removal" after verifying token lengths

## Setup

In [9]:
login()

In [10]:
torch.set_grad_enabled(False)

print(f"Loading model: {MODEL_NAME}")
model = TransformerBridge.boot_transformers(MODEL_NAME)
# model.enable_compatibility_mode()  # Uncomment if memory allows

Loading model: unsloth/Llama-3.2-1B


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

In [11]:
with open(DATASET_PATH) as f:
    dataset = json.load(f)

# Filter by type
dataset = [item for item in dataset if item["type"] in USE_TYPES]
N = len(dataset)
print(f"Dataset: {N} sets ({USE_TYPES})")

Dataset: 5 sets (['substitution'])


## Helpers

In [12]:
def find_measurement_pos(model, code_string):
    str_tokens = model.to_str_tokens(code_string)
    for i, tok in enumerate(str_tokens):
        if tok.strip() == '[' and i > 0 and '}' in str_tokens[i - 1]:
            return i + 1
    raise ValueError(f"Could not find measurement position in: {code_string[:100]}...")

In [13]:
def save_heatmap(data, title, filename, xaxis="Head", yaxis="Layer",
                 x_labels=None, y_labels=None):
    if isinstance(data, torch.Tensor):
        data = data.cpu().numpy()
    fig = px.imshow(
        data,
        labels=dict(x=xaxis, y=yaxis, color="Patching Effect"),
        x=x_labels,
        y=y_labels,
        title=title,
        color_continuous_midpoint=0,
        color_continuous_scale="RdBu",
    )
    fig.update_layout(width=max(800, data.shape[1] * 15), height=max(500, data.shape[0] * 25))
    filepath = OUTPUT_DIR / filename
    fig.write_image(str(filepath), scale=3)
    print(f"  Saved: {filepath}")
    return fig

## Main Pipeline

In [14]:
# Storage for aggregatable results (experiments 2 & 4)
all_attn_head_results = []       # Experiment 2
all_every_head_results = []      # Experiment 4

In [15]:
for idx, item in enumerate(dataset):
    print(f"\n{'='*60}")
    print(f"Processing {item['id']} ({idx+1}/{N}): {item['description']}")
    print(f"{'='*60}")

    # --- Tokenize ---
    clean_tokens = model.to_tokens(item["clean"])
    corrupted_tokens = model.to_tokens(item["corrupted"])

    if clean_tokens.shape != corrupted_tokens.shape:
        print(f"  SKIPPING: token length mismatch "
              f"(clean={clean_tokens.shape[1]}, corrupted={corrupted_tokens.shape[1]})")
        continue

    measurement_pos = find_measurement_pos(model, item["clean"])
    clean_target_id = model.to_single_token(item["clean_target"])
    corrupted_target_id = model.to_single_token(item["corrupted_target"])

    print(f"  Measurement position: {measurement_pos}")
    print(f"  Clean target: '{item['clean_target']}' (id={clean_target_id})")
    print(f"  Corrupted target: '{item['corrupted_target']}' (id={corrupted_target_id})")

    # --- Logit diff function (closed over this set's parameters) ---
    _mp = measurement_pos
    _ct = clean_target_id
    _crt = corrupted_target_id

    def get_logit_diff(logits, pos=_mp, ct=_ct, crt=_crt):
        pos_logits = logits[0, pos, :]
        return pos_logits[ct] - pos_logits[crt]

    # --- Forward passes ---
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

    clean_baseline = get_logit_diff(clean_logits).item()
    corrupted_baseline = get_logit_diff(corrupted_logits).item()
    print(f"  Clean logit diff: {clean_baseline:.4f}")
    print(f"  Corrupted logit diff: {corrupted_baseline:.4f}")

    if abs(clean_baseline - corrupted_baseline) < 0.01:
        print(f"  SKIPPING: logit diff too small to be meaningful")
        continue

    # --- Metric function ---
    _cb = corrupted_baseline
    _clb = clean_baseline

    def useeffect_metric(logits, cb=_cb, clb=_clb):
        return (get_logit_diff(logits) - cb) / (clb - cb)

    # --- Experiment 1: Residual stream patching (per-set heatmap) ---
    print("  Running Experiment 1: resid_pre patching...")
    resid_results = patching.get_act_patch_resid_pre(
        model, corrupted_tokens, clean_cache, useeffect_metric
    )
    str_tokens = model.to_str_tokens(clean_tokens[0])
    x_labels = [f"{tok} {i}" for i, tok in enumerate(str_tokens)]
    save_heatmap(
        resid_results, f"Exp1: resid_pre — {item['id']}",
        f"exp1_resid_pre_{item['id']}.png",
        xaxis="Position", yaxis="Layer", x_labels=x_labels
    )

    # --- Experiment 2: Attention head output patching (aggregatable) ---
    print("  Running Experiment 2: attn_head_out patching (all pos)...")
    attn_head_results = patching.get_act_patch_attn_head_out_all_pos(
        model, corrupted_tokens, clean_cache, useeffect_metric
    )
    all_attn_head_results.append(attn_head_results)
    save_heatmap(
        attn_head_results, f"Exp2: attn_head_out — {item['id']}",
        f"exp2_attn_head_{item['id']}.png",
    )

    # --- Experiment 3: Block-level patching (per-set heatmaps) ---
    print("  Running Experiment 3: block_every patching...")
    block_results = patching.get_act_patch_block_every(
        model, corrupted_tokens, clean_cache, useeffect_metric
    )
    block_labels = ["Residual Stream", "Attn Output", "MLP Output"]
    for b_idx, label in enumerate(block_labels):
        save_heatmap(
            block_results[b_idx],
            f"Exp3: {label} — {item['id']}",
            f"exp3_block_{label.replace(' ', '_').lower()}_{item['id']}.png",
            xaxis="Position", yaxis="Layer", x_labels=x_labels
        )

    # --- Experiment 4: Per-head every component patching (aggregatable) ---
    print("  Running Experiment 4: attn_head_all_pos_every patching...")
    every_head_results = patching.get_act_patch_attn_head_all_pos_every(
        model, corrupted_tokens, clean_cache, useeffect_metric
    )
    all_every_head_results.append(every_head_results)
    component_labels = ["Output", "Query", "Key", "Value", "Pattern"]
    for c_idx, label in enumerate(component_labels):
        save_heatmap(
            every_head_results[c_idx],
            f"Exp4: {label} — {item['id']}",
            f"exp4_head_{label.lower()}_{item['id']}.png",
        )

    # Clean up cache to free memory
    del clean_cache, corrupted_cache, clean_logits, corrupted_logits
    torch.cuda.empty_cache() if torch.cuda.is_available() else None


Processing sub_01 (1/5): fetch user — body: name→theme, array stays [name]
  Measurement position: 51
  Clean target: 'name' (id=609)
  Corrupted target: 'theme' (id=9224)
  Clean logit diff: -0.4448
  Corrupted logit diff: -6.5642
  Running Experiment 1: resid_pre patching...


  0%|          | 0/848 [00:00<?, ?it/s]

  Saved: llama3_1b_results/exp1_resid_pre_sub_01.png
  Running Experiment 2: attn_head_out patching (all pos)...


  0%|          | 0/512 [00:00<?, ?it/s]

  Saved: llama3_1b_results/exp2_attn_head_sub_01.png
  Running Experiment 3: block_every patching...


...

  0%|          | 0/512 [00:00<?, ?it/s]

  0%|          | 0/128 [00:00<?, ?it/s]

  0%|          | 0/128 [00:00<?, ?it/s]

  0%|          | 0/512 [00:00<?, ?it/s]

  Saved: llama3_1b_results/exp4_head_output_sub_05.png
  Saved: llama3_1b_results/exp4_head_query_sub_05.png
  Saved: llama3_1b_results/exp4_head_key_sub_05.png
  Saved: llama3_1b_results/exp4_head_value_sub_05.png
  Saved: llama3_1b_results/exp4_head_pattern_sub_05.png


In [16]:
if len(all_attn_head_results) > 0:
    print(f"\n{'='*60}")
    print(f"Aggregate results over {len(all_attn_head_results)} sets")
    print(f"{'='*60}")

    # --- Experiment 2: Average ---
    avg_attn = torch.stack(all_attn_head_results).mean(dim=0)
    save_heatmap(
        avg_attn,
        f"Exp2 Average: attn_head_out ({len(all_attn_head_results)} sets)",
        "exp2_attn_head_AVERAGE.png",
    )
    # Print top heads
    n_heads = avg_attn.shape[1]
    flat = avg_attn.flatten()
    top_indices = flat.abs().topk(min(10, flat.numel())).indices
    print("\n  Exp2 — Top 10 heads (averaged):")
    for rank, i in enumerate(top_indices):
        layer = i.item() // n_heads
        head = i.item() % n_heads
        print(f"    #{rank+1}: L{layer}H{head} = {avg_attn[layer, head]:+.4f}")

    # --- Experiment 4: Average ---
    avg_every = torch.stack(all_every_head_results).mean(dim=0)
    component_labels = ["Output", "Query", "Key", "Value", "Pattern"]
    for c_idx, label in enumerate(component_labels):
        save_heatmap(
            avg_every[c_idx],
            f"Exp4 Average: {label} ({len(all_every_head_results)} sets)",
            f"exp4_head_{label.lower()}_AVERAGE.png",
        )

    # Cross-type comparison for top heads
    print("\n  Exp4 — Top heads cross-component comparison (averaged):")
    top_heads_from_exp2 = []
    for i in flat.abs().topk(min(6, flat.numel())).indices:
        layer = i.item() // n_heads
        head = i.item() % n_heads
        top_heads_from_exp2.append((layer, head))

    header = f"  {'Head':>8s}" + "".join([f"{l:>10s}" for l in component_labels])
    print(header)
    for l, h in top_heads_from_exp2:
        row = f"  L{l}H{h:>5d}"
        for c_idx in range(len(component_labels)):
            row += f"{avg_every[c_idx, l, h].item():+10.4f}"
        print(row)

    # Save tensors for later analysis
    torch.save(avg_attn, OUTPUT_DIR / "exp2_attn_head_AVERAGE.pt")
    torch.save(avg_every, OUTPUT_DIR / "exp4_every_head_AVERAGE.pt")
    print(f"\n  Tensors saved to {OUTPUT_DIR}/")

print("\n✓ Pipeline complete.")


Aggregate results over 5 sets
  Saved: llama3_1b_results/exp2_attn_head_AVERAGE.png

  Exp2 — Top 10 heads (averaged):
    #1: L11H4 = -0.2601
    #2: L15H1 = +0.2286
    #3: L8H23 = +0.2010
    #4: L14H28 = +0.1985
    #5: L10H26 = +0.1931
    #6: L10H16 = +0.1820
    #7: L8H16 = +0.1631
    #8: L10H15 = -0.1626
    #9: L13H27 = -0.1551
    #10: L10H27 = -0.1488
  Saved: llama3_1b_results/exp4_head_output_AVERAGE.png
  Saved: llama3_1b_results/exp4_head_query_AVERAGE.png
  Saved: llama3_1b_results/exp4_head_key_AVERAGE.png
  Saved: llama3_1b_results/exp4_head_value_AVERAGE.png
  Saved: llama3_1b_results/exp4_head_pattern_AVERAGE.png

  Exp4 — Top heads cross-component comparison (averaged):
      Head    Output     Query       Key     Value   Pattern
  L11H    4   -0.2601   -0.0665   +0.0013   -0.0159   -0.2845
  L15H    1   +0.2286   +0.0432   -0.0005   +0.0074   +0.0240
  L8H   23   +0.2010   +0.0234   +0.0000   +0.0000   +0.0119
  L14H   28   +0.1985   +0.0806   +0.0000   +0.0000 